# Document Boundary Detection and Page Classification

## Objective
Uses Gemini to classify pages and group related pages into document sections inside a combined PDF file.

## Approach
- Extract text from each page
- Classify page/document type using Gemini
- Detect boundaries between documents
- Summarize results in a table

## Expected Result
This notebook represents the document-understanding stage of the externship pipeline, especially when a single PDF contains multiple document types.

## Running in Google Colab
These notebooks were developed in Google Colab. For reproducibility, place required PDFs in a `data/` folder when running locally, or upload them to the Colab working directory. The helper functions below try common Colab and GitHub-style paths.

## Security Note
API keys are not stored in the notebook. Use Colab Secrets with the name `GOOGLE_API_KEY` or set the environment variable manually.

## Project Context
This notebook is part of a curated document intelligence externship portfolio project completed through Outamation. The work focuses on OCR, document parsing, retrieval, LLM-based question answering, and prototype application development for mortgage-style document analysis.

## Data Note
The notebooks were originally developed in Google Colab. Any document files used for testing should be placed in the `data/` folder or uploaded directly into the Colab runtime. The sample documents used for this educational project do not contain sensitive personal information.


In [ ]:
# =========================
# Document Boundary Detection & Classification
# =========================

# Install dependencies
!pip -q install pymupdf google-genai pypdf pandas

import os
import re
import json
import fitz  # PyMuPDF
import time
from google.api_core import exceptions as genai_errors
import pandas as pd
from pypdf import PdfReader
from google import genai

# -------------------------
# PORTABLE COLAB/GITHUB HELPERS
# -------------------------
from pathlib import Path
import os

def resolve_path(filename_or_path):
    """Find a file in common Colab and GitHub project locations."""
    candidates = [
        Path(filename_or_path),
        Path("/content") / filename_or_path,
        Path("data") / Path(filename_or_path).name,
        Path("/content/data") / Path(filename_or_path).name,
    ]
    for path in candidates:
        if path.exists():
            return str(path)
    # Return GitHub-style path as the default so users know where to place data.
    return str(Path("data") / Path(filename_or_path).name)

def get_google_api_key():
    """Load GOOGLE_API_KEY from Colab Secrets or environment variables."""
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return os.getenv("GOOGLE_API_KEY")

# ---------
# SETTINGS
# ---------
GEMINI_API_KEY = get_google_api_key()

if not GEMINI_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found. Check Colab Secrets and turn Notebook access ON.")

client = genai.Client(api_key=GEMINI_API_KEY)
PDF_PATH          = resolve_path("Test Blob File.pdf")
GEMINI_MODEL_NAME = "gemini-2.5-flash"

# Free tier = 5 RPM → add a small pause between every call to stay under
CALL_DELAY_SECONDS = 13   # 60s / 5 RPM = 12s; use 13 to be safe


# -------------------------
# RATE-LIMIT-SAFE CALL WRAPPER
# -------------------------
def gemini_call(prompt: str, max_retries: int = 5) -> str:
    """Call Gemini with exponential backoff on 429 rate-limit errors."""
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL_NAME,
                contents=prompt
            )
            time.sleep(CALL_DELAY_SECONDS)   # throttle every successful call too
            return response.text.strip()
        except genai_errors.ClientError as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                # Parse suggested retry delay from error message if present
                match = re.search(r"retry in (\d+)", str(e))
                wait = int(match.group(1)) + 2 if match else (2 ** attempt) * 15
                print(f"  [RATE LIMIT] Attempt {attempt+1}/{max_retries} — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise   # re-raise non-quota errors immediately
    raise RuntimeError(f"Gemini call failed after {max_retries} retries.")

# -------------------------
# STEP 1: EXTRACT PAGE TEXT
# -------------------------
def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text("text").strip()
        pages.append((i, text))
    print(f"[INFO] Loaded {len(pages)} pages from {os.path.basename(pdf_path)}\n")
    return pages

# -------------------------
# STEP 2: CLASSIFY DOC TYPE
# -------------------------
def classify_document_type(page_text: str) -> str:
    prompt = f"""You are a document classifier. Based on the text below, identify the document type.
Return ONLY a short label such as: 'Fees Worksheet', 'Payslip', 'Employment Contract', 'Resume', 'Invoice', 'Mortgage Note', 'Unknown', etc.
No explanation — just the label.

Page text:
\"\"\"
{page_text[:2000]}
\"\"\"
Document type:"""
    return gemini_call(prompt)

# -------------------------
# STEP 3: BOUNDARY DETECTION
# -------------------------
def is_same_document(prev_text: str, curr_text: str) -> bool:
    prompt = f"""You are analyzing a multi-document PDF blob. Decide if the two pages below are part of the SAME document or the START of a NEW document.

Rules:
- Same document: continuation of content, same template/style, same parties, sequential sections.
- New document: different subject matter, different format/template, or clearly a new standalone document.

Respond with ONLY one word: YES (same document) or NO (new document).

--- PAGE A (previous page) ---
{prev_text[:1500]}

--- PAGE B (current page) ---
{curr_text[:1500]}

Same document? (YES/NO):"""
    answer = gemini_call(prompt).upper()
    return answer.startswith("Y")

# -------------------------
# STEP 4: MAIN PIPELINE
# -------------------------
pages    = extract_pages(PDF_PATH)
results  = []
doc_id   = 0
doc_type = None

for i, (page_num, text) in enumerate(pages):
    if i == 0:
        doc_type = classify_document_type(text)
        print(f"Page {page_num}: NEW doc  (doc_id={doc_id}) → {doc_type}")
    else:
        prev_text = pages[i - 1][1]
        same = is_same_document(prev_text, text)
        if same:
            print(f"Page {page_num}: same doc (doc_id={doc_id}) → {doc_type}")
        else:
            doc_id  += 1
            doc_type = classify_document_type(text)
            print(f"Page {page_num}: NEW doc  (doc_id={doc_id}) → {doc_type}")

    results.append({
        "page":     page_num,
        "doc_id":   doc_id,
        "doc_type": doc_type
    })

# -------------------------
# STEP 5: OUTPUT
# -------------------------
print("\n=== JSON Output ===")
print(json.dumps(results, indent=2))

print("\n=== DataFrame Output ===")
df = pd.DataFrame(results)
print(df.to_string(index=False))
